# **Cycle 4 Independent Notebook**

## *Isbella Bartoletti*

## 1. API and HTTP Model

In [42]:
%pip install pandas numpy

from pathlib import Path
import json
import pandas as pd
import numpy as np

#fixture = copy of API
DATA_DIR = Path("./")
FIXTURE_DIR = DATA_DIR / "api-fixtures"

assert FIXTURE_DIR.exists(), "Run from the teaching-notebook directory and keep datasets beside it."

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [43]:
# the two pokemon file paths

ability_path = FIXTURE_DIR / "battle-armor.json"
species_path = FIXTURE_DIR / "aegislash.json"

with ability_path.open(encoding="utf-8") as handle:
    ability = json.load(handle)

with species_path.open(encoding="utf-8") as handle:
    species = json.load(handle)

print(ability.keys())
print(ability["generation"].keys()) 

print(species.keys())
print(species["generation"].keys()) 

dict_keys(['id', 'name', 'is_main_series', 'generation', 'names', 'effect_entries', 'effect_changes', 'flavor_text_entries', 'pokemon'])
dict_keys(['name', 'url'])
dict_keys(['id', 'name', 'order', 'gender_rate', 'capture_rate', 'base_happiness', 'is_baby', 'is_legendary', 'is_mythical', 'hatch_counter', 'has_gender_differences', 'forms_switchable', 'growth_rate', 'pokedex_numbers', 'egg_groups', 'color', 'shape', 'evolves_from_species', 'evolution_chain', 'habitat', 'generation', 'names', 'pal_park_encounters', 'form_descriptions', 'flavor_text_entries', 'genera', 'varieties'])
dict_keys(['name', 'url'])


In [44]:
ENDPOINT_ARMOR = "https://pokeapi.co/api/v2/ability/battle-armor"

ENDPOINT_AEGISLASH = "https://pokeapi.co/api/v2/pokemon-species/aegislash"

# These are the specific values I'd like to investigate in ENDPOINT 2, aegishlash
EXPECTED_AEGISLASH_VALUES = {"name", "gender_rate", "capture_rate", "base_happiness"}

# for ENDPOINT 1, battle armor
EXPECTED_ABILITY_VALUES = {"names", "pokemon"}
EXPECTED_LENGTH = 11 # both lists (names, pokemon) have 11 items in the fixture

In [ ]:
# "protecting your secrets" - best practice: don't put API key directly in notebook

# validation for ENDPOINT 2 (aegislash)
def validate_species_payload(payload): 
    if not isinstance(payload, dict):
        raise TypeError("API payload must be a JSON object") # must be dictionary
    missing = EXPECTED_AEGISLASH_VALUES - set(payload) # expected keys minus keys actually in payload
    if missing: # true if any expected key is missing
        raise ValueError(f"Missing species fields: {sorted(missing)}") # checking if any fields in the keys list (from fixture) aren't in the live copy
        return payload


# validation for ENDPOINT 1 (battle-armor)
def validate_ability_payload(payload): 
    if not isinstance(payload, dict):
        raise TypeError("API payload must be a JSON object") # must be dictionary
    missing = EXPECTED_ABILITY_VALUES - set(payload) # expected keys minus keys actually in payload
    if missing: # true if any expected key is missing
        raise ValueError(f"Missing ability fields: {sorted(missing)}") # checking if any fields in the keys list (from fixture) aren't in the live copy

    #counts unique at any point
    # EX: new dictionary of {7,7,7,7} ---> set --> {7}

    # for each key in expected daily list
    lengths = {key: len(payload[key]) for key in EXPECTED_ABILITY_VALUES} # loop through keys in dict, value of each key number of values in the list
    if set(lengths.values()) != {EXPECTED_LENGTH}: # changed from unique lengths-set code bc this more logical, should be just {11}
        raise ValueError(f"Lists do not have the expected length {EXPECTED_LENGTH}: {lengths}") # checking if array in the payload has the same length
    return payload

validate_species_payload(species)
validate_ability_payload(ability)

{'id': 4,
 'name': 'battle-armor',
 'is_main_series': True,
 'generation': {'name': 'generation-iii',
  'url': 'https://pokeapi.co/api/v2/generation/3/'},
 'names': [{'name': 'カブトアーマー',
   'language': {'name': 'ja-hrkt',
    'url': 'https://pokeapi.co/api/v2/language/1/'}},
  {'name': '전투무장',
   'language': {'name': 'ko', 'url': 'https://pokeapi.co/api/v2/language/3/'}},
  {'name': '戰鬥盔甲',
   'language': {'name': 'zh-hant',
    'url': 'https://pokeapi.co/api/v2/language/4/'}},
  {'name': 'Armurbaston',
   'language': {'name': 'fr', 'url': 'https://pokeapi.co/api/v2/language/5/'}},
  {'name': 'Kampfpanzer',
   'language': {'name': 'de', 'url': 'https://pokeapi.co/api/v2/language/6/'}},
  {'name': 'Armadura Batalla',
   'language': {'name': 'es', 'url': 'https://pokeapi.co/api/v2/language/7/'}},
  {'name': 'Lottascudo',
   'language': {'name': 'it', 'url': 'https://pokeapi.co/api/v2/language/8/'}},
  {'name': 'Battle Armor',
   'language': {'name': 'en', 'url': 'https://pokeapi.co/api/v2

In [47]:
# For aegishlash
def acquire_aegishlash(use_live=False): # question for Ms Schott: why is user_live=False here, but True later on??
    source = "cached fixture"
    payload = species # setting payload equal to fixture, and if use_live is false then it just uses the fixture
    if use_live:
        try: # stops code excecution if an exception is raised in validate_weather_payload
            import requests
            response = requests.get(ENDPOINT_AEGISLASH, timeout=10) # sending a GET request using the request library, PokeAPI has no Header parameters
            response.raise_for_status() # checks the HTTP status code 
            # ex: 200 means OK, 404 not found, etc
            payload = response.json() # converts the response to JSON
            validate_species_payload(payload)
            source = response.url # URl of the source
        except (requests.RequestException, ValueError, TypeError) as exc:
            print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
            payload = species # live request unavailable, use fixture
    return validate_species_payload(payload), payload

acquisition_aegishlash_source = acquire_aegishlash(use_live=True)

# prints out raw data
acquisition_aegishlash_source

(None,
 {'id': 681,
  'name': 'aegislash',
  'order': 682,
  'gender_rate': 4,
  'capture_rate': 45,
  'base_happiness': 70,
  'is_baby': False,
  'is_legendary': False,
  'is_mythical': False,
  'hatch_counter': 20,
  'has_gender_differences': False,
  'forms_switchable': True,
  'growth_rate': {'name': 'medium',
   'url': 'https://pokeapi.co/api/v2/growth-rate/2/'},
  'pokedex_numbers': [{'entry_number': 681,
    'pokedex': {'name': 'national',
     'url': 'https://pokeapi.co/api/v2/pokedex/1/'}},
   {'entry_number': 119,
    'pokedex': {'name': 'kalos-central',
     'url': 'https://pokeapi.co/api/v2/pokedex/12/'}},
   {'entry_number': 332,
    'pokedex': {'name': 'galar',
     'url': 'https://pokeapi.co/api/v2/pokedex/27/'}},
   {'entry_number': 73,
    'pokedex': {'name': 'lumiose-city',
     'url': 'https://pokeapi.co/api/v2/pokedex/34/'}},
   {'entry_number': 681,
    'pokedex': {'name': 'champions',
     'url': 'https://pokeapi.co/api/v2/pokedex/36/'}}],
  'egg_groups': [{'name'

In [48]:
# For battle armor
def acquire_battle_armor(use_live=False):
    source = "cached fixture"
    payload = ability # setting payload equal to fixture, and if use_live is false then it just uses the fixture
    if use_live:
        try: # stops code excecution if an exception is raised in validate_weather_payload
            import requests
            response = requests.get(ENDPOINT_ARMOR, timeout=10) # sending a GET request using the request library, PokeAPI has no Header parameters
            response.raise_for_status() # checks the HTTP status code (200 means OK)
            payload = response.json() # converts the response to JSON
            validate_ability_payload(payload)
            source = response.url # URl of the source
        except (requests.RequestException, ValueError, TypeError) as exc:
            print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
            payload = ability # live request unavailable, use fixture
    return validate_ability_payload(payload), payload

acquisition_battle_armor_source = acquire_battle_armor(use_live=True)

# prints out raw data
acquisition_battle_armor_source

({'id': 4,
  'name': 'battle-armor',
  'is_main_series': True,
  'generation': {'name': 'generation-iii',
   'url': 'https://pokeapi.co/api/v2/generation/3/'},
  'names': [{'name': 'カブトアーマー',
    'language': {'name': 'ja-hrkt',
     'url': 'https://pokeapi.co/api/v2/language/1/'}},
   {'name': '전투무장',
    'language': {'name': 'ko',
     'url': 'https://pokeapi.co/api/v2/language/3/'}},
   {'name': '戰鬥盔甲',
    'language': {'name': 'zh-hant',
     'url': 'https://pokeapi.co/api/v2/language/4/'}},
   {'name': 'Armurbaston',
    'language': {'name': 'fr',
     'url': 'https://pokeapi.co/api/v2/language/5/'}},
   {'name': 'Kampfpanzer',
    'language': {'name': 'de',
     'url': 'https://pokeapi.co/api/v2/language/6/'}},
   {'name': 'Armadura Batalla',
    'language': {'name': 'es',
     'url': 'https://pokeapi.co/api/v2/language/7/'}},
   {'name': 'Lottascudo',
    'language': {'name': 'it',
     'url': 'https://pokeapi.co/api/v2/language/8/'}},
   {'name': 'Battle Armor',
    'language': 

### **Key Description for Aegishlash *(I only do a select few because the number of keys is quite extensive)***

- `id`: int, the species unique ID in PokeAPI
- `name`: str, the aegishlash species name
- `base_happiness`: int, friendship value
- `is_baby`: bool, whether it's a baby Pokemon
- `is_legendary`: bool, whether it's a legendary Pokemon
- `capture_rate`: int, how easy species is to catch (higher value means easier)

